# Import data
データ取り込みを行う。 PostgresSQL PgVector 想定。

In [1]:
import os
import dotenv
from langchain_postgres import PGEngine

dotenv.load_dotenv()
CONNECTION_STRING = os.getenv("ENV_PG_CONNECTION_STRING")

# Tenki

In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("../docs/dataset_news/", loader_cls=TextLoader)
docs_origin = loader.load()

In [ ]:
from agent_assistant.utils.chunker.markdown import MarkdownHeaderChunker
from agent_assistant.utils.documentstore.markdown import MarkdownDocumentStore
from agent_assistant.utils.retriever import MonthlyNewsChunkStore

engine = PGEngine.from_connection_string(url=CONNECTION_STRING)
doc_cs = MonthlyNewsChunkStore(engine)
doc_ds = MarkdownDocumentStore("monthly_news", doc_cs, doc_cs, MarkdownHeaderChunker())
doc_ds.connect()

In [ ]:
# 誤実行でドキュメント重複しないようコメントアウトしておく
# doc_ds.import_documents(docs_origin)

In [5]:
results = doc_ds.search_documents("databricks", top_k=5)
# メタデータ フィールドを使用してベクトル検索をフィルタする
# results = vectorstore.similarity_search("maroon puffer jacket", filter={"price": {"$lt": 200.0}})

# results
tmpl = """
----------------------------
{id}
----------------------------
{content}
"""

for item in results:
    print(tmpl.format(id=item.metadata["source"], content=item.page_content[:100]))


----------------------------
2025-12_01.md
----------------------------
## テクノロジーニュース  
**MLflow 3.8.0とDatabricks Runtime 18.0のリリース:**
12月、Databricksは年内最後の大型アップデートを行いました。ML


----------------------------
2025-05_01.md
----------------------------
## テクノロジーニュース  
**Databricks Data + AI Summit 2025 開幕:**
今年のData + AI Summitは、生成AIの「実用化」から「エージェント化」へ


----------------------------
2025-09_01.md
----------------------------
## テクノロジーニュース  
**Databricksプラットフォームの包括的アップデート:**
9月、Databricksはプラットフォーム全体にわたるユーザビリティの向上と機能強化を実施しました


----------------------------
2025-11_01.md
----------------------------
## テクノロジーニュース  
**DatabricksにおけるModel Context Protocol (MCP) のサポート:**
Databricks Marketplaceで「Model 


----------------------------
2025-01_01.md
----------------------------
## テクノロジーニュース  
**MLflow 3.0の発表とGenAI機能の強化:**
Databricks Data + AI Summit 2025において、MLflow 3.0が発表されまし



# Obsidian Vault

In [ ]:
from pathlib import Path
from sqlalchemy import create_engine

from agent_assistant.utils.chunker.text import TextChunker
from agent_assistant.utils.documentstore.obsidian import ObsidianDocumentStore
from agent_assistant.utils.retriever import ObsidianChunkStore

vault_path = Path("../docs/dataset_obsidian/")
# エンジン初期化
sa_engine = create_engine(CONNECTION_STRING)
pg_engine = PGEngine.from_connection_string(CONNECTION_STRING, pool_size=5)
chunk_store = ObsidianChunkStore(pg_engine)
obsidian_store = ObsidianDocumentStore(
    "obsidian_vault", chunk_store, sa_engine, TextChunker(chunk_size=300, chunk_overlap=30)
)

In [7]:
from agent_assistant.obsidian import VaultLoader

loader = VaultLoader(vault_path)
docs = loader.load()

print(f"{len(docs)} 件のノートを読み込みました")
print(docs[0])

98 件のノートを読み込みました
page_content='# 概要

Databricks でのシステム開発を支援するベースを作る

# 機能

- ディレクトリ構造の提示
- 設定ファイル取得機能の提供
- ロギング機構の提供
- ETL支援
    - bundle 構成
    - パイプラインサンプル & マニュアル
    - RAG 用のドキュメント取り込みテンプレート
    - Bronze の増分取り込み。Silver へのバッチ基準日単位での洗い替え
- LLM
    - bundle 構成
    - playground サンプル
    - RAG Retreiver 改造基盤 (Vector Search Index、Chunking、Book RAG、プロンプト)
        - Embbeding モデルの切り替え
        - Chunking サイズ、スライド量の切り替え
        - Book RAG 実装
        - プロンプトに応じてRAG対象となるドキュメントソースを推測するルーターの実装
    - MLFLowによる性能評価を行うサンプル

# 関連ノート

- [[AI エージェント開発]]' metadata={'source': 'Project Databricks Utility for ETL & AI.md', 'path': '01_Inbox/Project Databricks Utility for ETL & AI.md', 'created': '2026-03-07T05:13:11.404316', 'last_modified': '2026-01-01T13:01:10.533918', 'last_accessed': '2026-03-07T05:13:47.027719', 'date': '2025-12-14 10:18:05', 'tags': 'データ分析,focused/LLM,focused/プログラミング', 'forward_links': ['03_Structure/AI エージェント開発.md']}


In [8]:
obsidian_store.connect()
obsidian_store.import_documents(docs)

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from agent_assistant.utils.chunker.text import TextChunker

vault_path = Path("../docs/dataset_obsidian/")
note_path = vault_path.resolve() / "02_Daily/2025-12-28.md"

TextChunker().chunk([Document(note_path.read_text())])

[Document(metadata={'start_index': 0}, page_content='---\ndate: 2025-12-28 13:49:23\ntags:\n---\n# TODO\n\n- [x] mlflow evaluate\n- [x] google ai studio 契約\n- [x] 健康診断\n\n# 調べもの\n## Claude on Databricks on AWS\n\nDatabricks の基盤モデルとして使用できる Claude との通信はインターネット上へ出ていると思われる。\nDatabricks が Anthropic に対する窓口とされている。'),
 Document(metadata={'start_index': 243}, page_content='> **How will Anthropic Foundation Model Serving be charged?**\n> Usage of these services will appear on your bill under the Anthropic Model Serving SKU. This SKU is global\n\nFoundation Model APIを提供する一環として、Databricksはあなたのデータを元の地域やクラウドプロバイダーの外で処理することがあるとされている。\n\n> **Data processing and residency**'),
 Document(metadata={'start_index': 537}, page_content='> As part of providing the Foundation Model APIs, Databricks might process your data outside of the region and cloud provider where your data originated.'),
 Document(metadata={'start_index': 692}, page_content='[Data security when using Databricks Foundation Mod... - Databri

In [10]:
obsidian_store._chunk_vs.similarity_search_with_relevance_scores("プロンプトエンジニアリング", k=5)

[(Document(id='c15ab517-9854-4442-99ea-beab842c09aa', metadata={'source': '2025-12-30.md', 'created': '2026-03-07T05:13:11.444260', 'last_modified': '2026-02-02T18:38:09.301950', 'last_accessed': '2026-03-07T05:13:47.027719', 'date': '2025-12-30 10:47:10', 'tags': 'None', 'forward_links': ['05_Note/SSH.md'], 'path': '02_Daily/2025-12-30.md', 'start_index': 544}, page_content='プロンプトエンジニアリングの機能の利用は見合わせる。 使用する事で得られるメリットが増える手間のデメリットに対して見合わない事、使われている geteway 機能の今後が不明瞭である事や、資料ページの画面がv2のものである事から、開発元が機能整理やブラシュアップをしてから触れた方が良いと判断。'),
  0.7712522285430309),
 (Document(id='48c91486-df82-49c6-829a-33d8941b0ddc', metadata={'source': 'Anthropic AIエージェントのための効果的なコンテキストエンジニアリング.md', 'created': '2026-03-07T05:13:11.733854', 'last_modified': '2026-01-12T00:10:13.049175', 'last_accessed': '2026-03-07T05:13:47.027719', 'URL': 'https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents', 'date': '2026-01-12 08:52:09', 'tags': 'focused/LLM', 'forward_links': [], 'path': '04_Literature/A

In [34]:
from IPython.display import display, Markdown

results = obsidian_store.search_documents(
    "コンピュータサイエンス",
    search_type="similarity_score_threshold",
    k=3,
    score_threshold=0.65,
)
tmpl = """
------------------------------
path: {path}
------------------------------
{content}
"""
for doc in results:
    md_text = tmpl.format(
        path=doc.metadata.get("path"), content=(doc.page_content[:300] + " ...")
    )
    display(Markdown(md_text))


------------------------------
path: 03_Structure/システム開発手法.md
------------------------------
# 開発工程

- [[開発 (ETL)]]
- [[運用設計の教科書]]

# ビジネススキル

- [[エンジニアを説明上手にする本 - 相手に応じた技術情報や知識の伝え方]]
- [[一気に即戦力！学生と社会人新人のためのロジカルシンキング&ライティング講座]]
- [[新任リーダーのためのITプロジェクト管理入門]]

- 人に指示する際に LLM にチェックさせた方が良い ([[2026-02-01#プロンプトメモ]])

# ツール使いこなし

- [[VSCode]]
- [[Chrome]]
- [[Gemini CLI]]
- [[SSH]]
- [[Postgres SQL ...



------------------------------
path: 02_Daily/2026-01-04.md
------------------------------
# ToDo

- [ ] agent 開発
- [ ] AWS SA Pro 試験について調べる

# 調べもの
### コンテキストエンジニアリング

現時点 (2026/01/04) で提案されている手法のリンク集。後で読みたい
- [コンテキストエンジニアリング超まとめ｜数理の弾丸 ─ 京大博士のAI解説](https://note.com/mathbullet/n/n6cc01bcfa868)
- [Context Engineering](https://blog.langchain.com/context-engineering-for-agents/)
- [[2507.133 ...



------------------------------
path: 02_Daily/2026-02-07.md
------------------------------
# 調べもの
## VSCode Databricks 拡張機能でエラー

ノート作成済み: [[Databricks 基本#ケース Spark 実行でエラーが発生]]
python ファイルをデバッグ実行する分にはエラーは発生しない (2026-02-07 時点)。 ipynb ファイル実行時に発生する。

```python
The above exception was the direct cause of the following exception:

ValueError                                Traceback (most recent ...
